# Cambodian ALPR — IMPROVEMENT #1: STN + *realistic* synthetic data

**What we already have (measured on 149 real test crops):**

| model | upright | upside-down |
|---|---|---|
| `crnn_finetuned.pth` (deployed) | 80.5% | 0% |
| `crnn_stn2.pth` (current best) | 79.2% | **55%** |

**What this run tries (#1):** the synthetic plate generator was upgraded to look much more
like real photos — blue plate ink, blue border/underline, camera perspective, uneven
lighting, blur, sensor noise and JPEG artefacts. Same STN, same recipe, *better fake data*.

**The number to beat is 55% upside-down, while upright stays around 79%.**

**Before you run:** make sure the zip in Drive/`ALPR` is the CURRENT one. Cell 3 checks this
for you and will tell you loudly if it is stale. Then: **Runtime → Change runtime type → T4 GPU**,
and run top to bottom. Takes roughly 1.5–2.5 hours.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BUNDLE = '/content/drive/MyDrive/ALPR/alpr_colab_bundle.zip'   # edit if needed

import os, zipfile
os.makedirs('/content/alpr', exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall('/content/alpr')
%cd /content/alpr

In [ ]:
# ---- BUNDLE FRESHNESS CHECK -- do not skip -------------------------------
# Proves the zip you uploaded contains BOTH improvements. If either line says
# MISSING, the Drive zip is an old one: rebuild it on the laptop with
#     python scripts/tools/make_colab_bundle.py
# re-upload (replacing the old file), then re-run the cell above.
gen  = open('scripts/recognition/generate_synthetic.py', encoding='utf-8').read()
crnn = open('src/recognition/crnn_model.py', encoding='utf-8').read()

ok_stn   = 'class STN' in crnn
ok_synth = '_realistic_augment' in gen

print('STN straightening layer   :', 'PRESENT' if ok_stn   else '*** MISSING ***')
print('realistic synthetic (#1)  :', 'PRESENT' if ok_synth else '*** MISSING ***')
assert ok_stn and ok_synth, 'STALE BUNDLE -- rebuild and re-upload before continuing.'
print('Bundle is current. Safe to continue.')


In [ ]:
!pip -q install ultralytics opencv-python-headless pyyaml tqdm pillow matplotlib

In [ ]:
# Generate a BIG synthetic set with the IMPROVED (realistic) renderer.
# --rotate180 flips ~half of them during training, so the STN sees roughly
# 8,000 upside-down examples instead of the ~340 real ones.
!python scripts/recognition/generate_synthetic.py --train 20000 --valid 1500 --test 800

In [ ]:
# Eyeball the new synthetic plates -- they should look like PHOTOS of plates,
# not clean computer graphics. If they still look flat/clean, #1 did not apply.
import glob
from PIL import Image
import matplotlib.pyplot as plt

paths = sorted(glob.glob('data/synthetic/train/*.jpg'))[:8]
fig, axes = plt.subplots(4, 2, figsize=(10, 6))
for ax, p in zip(axes.ravel(), paths):
    ax.imshow(Image.open(p)); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# BASELINE 1 -- the deployed upright-only reader (expect ~80% up / ~0% down)
!python scripts/tools/test_rotation_reading.py --weights models/recognition/crnn_finetuned.pth

In [ ]:
# BASELINE 2 -- crnn_stn2.pth, the CURRENT BEST (expect ~79% up / ~55% down).
# This is the real number to beat. It is not in the bundle; copy it from Drive.
import os, shutil
src = '/content/drive/MyDrive/ALPR/trained/crnn_stn2.pth'
if os.path.exists(src):
    shutil.copy(src, 'models/recognition/crnn_stn2.pth')
    !python scripts/tools/test_rotation_reading.py --weights models/recognition/crnn_stn2.pth
else:
    print('crnn_stn2.pth not found in Drive/ALPR/trained -- upload it, or use the')
    print('known measured baseline instead: upright 79.2%, upside-down 55.0%.')

In [ ]:
# THE EXPERIMENT -- same recipe as crnn_stn2, but fed the realistic synthetic data.
!python scripts/recognition/finetune_crnn.py --stn --rotate180 0.5 --synth-n 16000     --out models/recognition/crnn_stn3.pth --epochs 80

In [ ]:
# MEASURE -- the honest verdict, same 149 held-out real crops as every baseline above.
!python scripts/tools/test_rotation_reading.py --weights models/recognition/crnn_stn3.pth

### How to read the result

Compare **crnn_stn3** against **crnn_stn2 (79.2% up / 55.0% down)**:

| what you see | what it means | what to do |
|---|---|---|
| upside-down **> 55%**, upright **≥ ~77%** | #1 worked — realistic data closed part of the fake→real gap | keep `crnn_stn3.pth`, then do #2 |
| upside-down **≈ 55%** (±3%) | #1 made no real difference — the gap was not the realism | keep `crnn_stn2.pth`, then do #2 |
| upside-down up but upright **< 77%** | it traded away the normal case — not an improvement | keep `crnn_stn2.pth`, then do #2 |
| upside-down **< 55%** | the new degradation is too harsh | keep `crnn_stn2.pth`, then do #2 |

A ±3% wobble is roughly 4 crops out of 149 — that is noise, not a result. Either way, **#2
(more real labelled crops) comes next**, because that is the stronger lever.

Copy the whole printed block back to Claude — do not summarise it.

In [ ]:
import shutil, os
os.makedirs('/content/drive/MyDrive/ALPR/trained', exist_ok=True)
if os.path.exists('models/recognition/crnn_stn3.pth'):
    shutil.copy('models/recognition/crnn_stn3.pth', '/content/drive/MyDrive/ALPR/trained/')
    print('saved crnn_stn3.pth -> Drive/ALPR/trained')
else:
    print('crnn_stn3.pth was not produced -- training failed, check the cell above.')
print('Next: download it into the laptop at models/recognition/, then re-measure on Windows.')